In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from datetime import datetime

warnings.filterwarnings("ignore")

# --- Configuration ---
BS4_DATE = '2017-04-01'
METRO_DATE = '2017-06-17'
TXT_OUTPUT_FILENAME = 'ITS_Advanced_Regression_Summary.txt'
OUTPUT_DIR = '/content/its_outputs_advanced_new'

os.makedirs(OUTPUT_DIR, exist_ok=True)
summary_dir = os.path.join(OUTPUT_DIR, 'model_summaries')
plots_dir = os.path.join(OUTPUT_DIR, 'plots')
os.makedirs(summary_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)

# --------- 1. Data Preparation Functions ----------

def prepare_daily_data(df, date_col, prefix=''):
    df.rename(columns={date_col: 'Date'}, inplace=True)
    df['Date'] = pd.to_datetime(df['Date'], format='mixed', errors='coerce').dt.normalize()
    col_map = {col: f'{prefix}{col}' for col in df.columns if col not in ['Date']}
    df.rename(columns=col_map, inplace=True)
    return df

def prepare_annual_data(df):
    df.rename(columns={'Date': 'Financial_Year_Str'}, inplace=True)
    # Select Total Non-transport vehicles (private)
    df_annual = df.loc[df['Category'].str.contains('Total Non-transport', na=False),
                         ['Financial_Year_Str', 'Total registered vehicles']].copy()

    df_annual['End_Year'] = df_annual['Financial_Year_Str'].str.split('-').str[-1].astype(int) + 2000
    df_annual['Date'] = pd.to_datetime(df_annual['End_Year'].astype(str) + '-03-31')

    df_annual.rename(columns={'Total registered vehicles': 'Annual_Vehicle_Total'}, inplace=True)

    return df_annual[['Date', 'Annual_Vehicle_Total']]


# --------- 2. Load and Prepare All Data ----------

try:
    # Daily data (Assuming files are in /content/)
    majestic_pollutants = prepare_daily_data(pd.read_csv('/content/majestic_pollutants_cleaned_2016_2019.csv'), 'Timestamp')
    majestic_weather = prepare_daily_data(pd.read_csv('/content/majestic_weather_final.csv'), 'date', 'weather_')
    peenya_pollutants = prepare_daily_data(pd.read_csv('/content/new_peenya_pollutants_cleaned_2016_2019.csv'), 'Timestamp', 'peenya_')
    vehicle_annual = prepare_annual_data(pd.read_csv('/content/bs4_registration.csv'))
except FileNotFoundError as e:
    print(f"Error: One of the required CSV files was not found: {e}")
    # Using dummy data for demonstration if files are not present in exact path
    majestic_pollutants = prepare_daily_data(pd.DataFrame({'Timestamp': pd.date_range('2016-01-01', periods=1000), 'PM10 (µg/m³)': np.random.rand(1000)*100, 'NO (µg/m³)': np.random.rand(1000)*50, 'PM2.5 (µg/m³)': np.random.rand(1000)*50, 'NO2 (µg/m³)': np.random.rand(1000)*50, 'CO (mg/m³)': np.random.rand(1000)*5}), 'Timestamp')
    majestic_weather = prepare_daily_data(pd.DataFrame({'date': pd.date_range('2016-01-01', periods=1000), 'temp_mean': np.random.rand(1000)*30, 'wind_speed': np.random.rand(1000)*5}), 'date', 'weather_')
    peenya_pollutants = prepare_daily_data(pd.DataFrame({'Timestamp': pd.date_range('2016-01-01', periods=1000), 'PM2.5 (µg/m³)': np.random.rand(1000)*80, 'NO2 (µg/m³)': np.random.rand(1000)*40, 'CO (mg/m³)': np.random.rand(1000)*5, 'NO (µg/m³)': np.random.rand(1000)*40}), 'Timestamp', 'peenya_')
    vehicle_annual = prepare_annual_data(pd.DataFrame({'Date': ['2016-17', '2017-18', '2018-19', '2019-20'], 'Category': ['Total Non-transport']*4, 'Total registered vehicles': [5000000, 6000000, 7000000, 8000000]}))

# Rename daily pollutant columns for merging and formula use
majestic_pollutants.rename(columns={'PM10 (µg/m³)': 'Majestic_PM10', 'NO (µg/m³)': 'Majestic_NO',
                                    'NO2 (µg/m³)': 'Majestic_NO2', 'CO (mg/m³)': 'Majestic_CO',
                                    'PM2.5 (µg/m³)': 'Majestic_PM25'}, inplace=True)
peenya_pollutants.rename(columns={'peenya_PM2.5 (µg/m³)': 'Peenya_PM25', 'peenya_NO2 (µg/m³)': 'Peenya_NO2',
                                  'peenya_CO (mg/m³)': 'Peenya_CO', 'peenya_NO (µg/m³)': 'Peenya_NO'}, inplace=True)

# Select relevant weather columns
weather_expected = ['temp_mean','wind_speed']
weather_cols_map = {f'weather_{col}': col for col in weather_expected}
majestic_weather = majestic_weather.rename(columns=weather_cols_map)[list(weather_expected) + ['Date']]

# --------- 3. Merge Daily Data and Annual Vehicle Data ----------

# Merge daily data
df = majestic_pollutants.merge(peenya_pollutants, on='Date', how='inner').merge(majestic_weather, on='Date', how='inner')

# Merge vehicle total by year and scale
df['Year'] = df['Date'].dt.year
vehicle_annual['Year'] = vehicle_annual['Date'].dt.year

year_vehicle_map = vehicle_annual.set_index('Year')['Annual_Vehicle_Total'].to_dict()
# Fill forward and backward to cover all dates
df['Annual_Vehicle_Total'] = df['Year'].map(year_vehicle_map).fillna(method='ffill').fillna(method='bfill')

df['Vehicle_Total_Scaled'] = (df['Annual_Vehicle_Total'] - df['Annual_Vehicle_Total'].mean()) / df['Annual_Vehicle_Total'].std()


# --------- 4. Create ITS Variables for Multiple Interventions ----------
df.sort_values(by='Date', inplace=True)
df.reset_index(drop=True, inplace=True)

# T: Time since start of series (used as pre-intervention slope)
df['T'] = (df['Date'] - df['Date'].min()).dt.days

# BS4 Intervention
df['BS4_P'] = (df['Date'] >= BS4_DATE).astype(int)
df['BS4_TP'] = df['T'] * df['BS4_P']

# Metro Intervention
df['Metro_P'] = (df['Date'] >= METRO_DATE).astype(int)
df['Metro_TP'] = df['T'] * df['Metro_P']


# --------- 5. Final Data Cleaning and Selection ----------

df = df.set_index('Date').sort_index()

# Interpolate and drop NaNs (Good practice before modeling)
df = df.interpolate(method='time').ffill().bfill()
df_its = df.dropna().copy()

# Reset index to make 'Date' a column again for easy plotting/variable access
df_its = df_its.reset_index()


# Check which outcomes and confounders are present
pollutants = [col for col in ['Majestic_PM10', 'Majestic_PM25', 'Majestic_NO', 'Majestic_NO2', 'Majestic_CO'] if col in df_its.columns]
industrial_vars = [c for c in ['Peenya_PM25', 'Peenya_NO', 'Peenya_NO2', 'Peenya_CO'] if c in df_its.columns]
weather_vars = ['temp_mean', 'wind_speed']

if not pollutants:
    raise ValueError("No valid pollutant columns found after cleaning. Check data/renaming.")


# --------- 6. Run ITS Regression Models (Advanced/Robust Fit) ----------

models = {}

for pollutant in pollutants:
    # Build formula string dynamically
    current_industrial_vars = [v for v in industrial_vars if v in df_its.columns]

    rhs_terms = ['T', 'Metro_P', 'Metro_TP', 'BS4_P', 'BS4_TP', 'Vehicle_Total_Scaled'] + current_industrial_vars + weather_vars
    formula = f"{pollutant} ~ " + ' + '.join(rhs_terms)

    # Fit model with HC1 (Robust) Standard Errors
    model = smf.ols(formula=formula, data=df_its).fit(cov_type='HC1')
    models[pollutant] = model

    # Print & save summary
    print("\n" + "="*60)
    print(f"ITS Results (Robust Model) for {pollutant}")
    print("="*60)
    print(model.summary())

    with open(os.path.join(summary_dir, f"{pollutant}_its_robust_summary.txt"), 'w') as f:
        f.write(str(model.summary()))

# --------- 7. Visualization: Observed, Fitted, Counterfactual (Original Code Plotting Style) ----------

metro_intervention_date = pd.to_datetime(METRO_DATE)
bs4_confounder_date = pd.to_datetime(BS4_DATE)

for pollutant, model in models.items():

    # Predict the Fitted Line (Actual Scenario)
    fitted = model.predict(df_its)

    # Predict the Counterfactual (No Metro Effect)
    cf_data = df_its.copy()
    cf_data['Metro_P'] = 0
    cf_data['Metro_TP'] = 0
    counterfactual_all = model.predict(cf_data)

    # Prepare data index for plotting
    plot_index = df_its['Date']

    plt.figure(figsize=(12, 6))

    # Observed (daily raw data)
    plt.plot(plot_index, df_its[pollutant], label='Observed (Daily)', alpha=0.6, linewidth=0.5, color='gray')

    # Fitted line (model prediction on original data)
    plt.plot(plot_index, fitted, label='ITS Robust Fit', linewidth=2, color='darkgreen')

    # We only plot counterfactual in the post-intervention period for clarity
    counterfactual_post = counterfactual_all.copy()
    # Masking the pre-intervention period with NaN for the counterfactual plot line
    counterfactual_post[plot_index < metro_intervention_date] = np.nan

    plt.plot(plot_index, counterfactual_post, label='Counterfactual (No Metro Effect)', linestyle='--', linewidth=2, color='red')

    # Mark intervention & confounder dates
    plt.axvline(x=metro_intervention_date, color='green', linestyle='--', linewidth=1.5, label=f'Metro Opening: {metro_intervention_date.date()}')
    plt.axvline(x=bs4_confounder_date, color='orange', linestyle=':', linewidth=1.2, label=f'BS4 Start: {bs4_confounder_date.date()}')

    plt.title(f'ITS Analysis (Daily Data) for {pollutant} - Robust Model')
    plt.xlabel('Date')
    plt.ylabel(f'{pollutant} concentration')
    plt.legend(loc='best')
    plt.grid(alpha=0.25)
    plt.tight_layout()

    # Save plot
    plotfile = os.path.join(plots_dir, f'{pollutant}_its_plot_original_style.png')
    plt.savefig(plotfile, dpi=150)
    plt.close()

# --------- 8. Save final merged dataset ----------
df_its.to_csv(os.path.join(OUTPUT_DIR, 'its_merged_data_final.csv'), index=False)

print("\nAll done.")
print(f"Outputs saved in: {OUTPUT_DIR}")
print(f"- model summaries: {summary_dir}")
print(f"- plots: {plots_dir}")


ITS Results (Robust Model) for Majestic_PM10
                            OLS Regression Results                            
Dep. Variable:          Majestic_PM10   R-squared:                       0.157
Model:                            OLS   Adj. R-squared:                  0.150
Method:                 Least Squares   F-statistic:                     34.59
Date:                Tue, 18 Nov 2025   Prob (F-statistic):           5.43e-71
Time:                        06:44:41   Log-Likelihood:                -7703.7
No. Observations:                1461   AIC:                         1.543e+04
Df Residuals:                    1448   BIC:                         1.550e+04
Df Model:                          12                                         
Covariance Type:                  HC1                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------

In [ ]:
# Create a compressed zip file of the output folder
!zip -r /content/its_outputs.zip /content/its_outputs_advanced_new/

# Download the zip file to your local machine
from google.colab import files
files.download('/content/its_outputs.zip')

  adding: content/its_outputs/ (stored 0%)
  adding: content/its_outputs/plots/ (stored 0%)
  adding: content/its_outputs/plots/Majestic_NO2_its_plot.png (deflated 3%)
  adding: content/its_outputs/plots/Majestic_CO_its_plot.png (deflated 4%)
  adding: content/its_outputs/plots/Majestic_NO_its_plot.png (deflated 4%)
  adding: content/its_outputs/plots/Majestic_PM10_its_plot.png (deflated 5%)
  adding: content/its_outputs/its_merged_data_raw.csv (deflated 62%)
  adding: content/its_outputs/model_summaries/ (stored 0%)
  adding: content/its_outputs/model_summaries/Majestic_PM10_estimates.txt (deflated 40%)
  adding: content/its_outputs/model_summaries/Majestic_CO_its_summary.txt (deflated 68%)
  adding: content/its_outputs/model_summaries/Majestic_CO_estimates.txt (deflated 41%)
  adding: content/its_outputs/model_summaries/Majestic_NO2_its_summary.txt (deflated 65%)
  adding: content/its_outputs/model_summaries/Majestic_NO2_estimates.txt (deflated 40%)
  adding: content/its_outputs/mode

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>